# Pipeline: train → SHAP → RAG → action plan → scoring → apply

One notebook for the offline paper path. Each stage calls the script that owns that work (`scripts/pipeline.py`). You do not need to open the specialist notebooks to run it.

| Step | What it does | Script | Writes |
|------|----------------|--------|--------|
| 1 | Train 3-party VFL + meta-classifier | `scripts/detect_train.py` | `model/` |
| 2 | Predict class + KernelSHAP domain shares | `scripts/detect_predict.py` | `outputs/predictions_detailed_*.json` |
| 3 | Build FAISS policy index | `scripts/rag_build.py` | `RAG_docs/vector_store/` |
| 4 | RAG + LLM mitigation plans | `scripts/reason.py` | `RAG_docs/action_plans/` |
| 5 | BERTScore / BLEU / ROUGE on those plans | `scripts/evaluate.py` | `RAG_docs/action_plans/scoring_results.json` |
| 6 | Whitelist + plan-binding (no chain) | `scripts/apply_gates.py` | prints only |
| demo | Tiny stdlib Detect→Reason→Apply | `scripts/demo_pipeline.py` | prints only |

**How to run**

1. Kernel cwd can be the repo root, `backend/`, or `backend/notebooks/`. The setup cell finds `backend/` and then uses the repo root (same as `python scripts/pipeline.py`).
2. The **demo** flag is on by default (no PyTorch, no FAISS, no API, no blockchain).
3. Turn on `RUN_DETECT_TRAIN`, `RUN_DETECT_PREDICT`, `RUN_RAG_INDEX`, `RUN_REASON`, and `RUN_EVALUATE` when `datasets/`, `RAG_docs/knowledge/`, a saved `model/`, and `OPENAI_API_KEY` are ready. Run the cells in order: train before SHAP, SHAP before plans, plans before scoring.

Blockchain commit (`AgenticTrustRegistry`) is not in this notebook. Live HTTP end-to-end is `python run/attack_monitor.py` or `python scripts/pipeline.py e2e`.

## 0. Setup

In [ ]:
import os
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_backend = None
for cand in (_cwd, _cwd.parent, _cwd / "backend", _cwd.parent / "backend"):
    if (cand / "scripts" / "vfl.py").is_file() and (cand / "app" / "main.py").is_file():
        _backend = cand
        break
if _backend is None:
    raise FileNotFoundError("Could not find backend/ (expected scripts/vfl.py and app/main.py)")
if str(_backend) not in sys.path:
    sys.path.insert(0, str(_backend))

from scripts.env import ensure_backend_on_sys_path, load_project_dotenv

BACKEND = ensure_backend_on_sys_path()
REPO = BACKEND.parent
os.chdir(REPO)
load_project_dotenv()
print("backend:", BACKEND)
print("repo cwd:", Path.cwd())

In [ ]:
# Heavy stages stay off until datasets, model/, RAG_docs, and an API key are in place.
RUN_DETECT_TRAIN = False
RUN_DETECT_PREDICT = False
RUN_RAG_INDEX = False
RUN_REASON = False
RUN_EVALUATE = False
RUN_APPLY_DEMO = True
RUN_STDLIB_DEMO = True

from scripts.pipeline import STAGES, run_stage

print("stages:")
for name, spec in STAGES.items():
    print(f"  {name:<16} {spec['target']}")

## 0b. Shared modules

This notebook imports the live modules. It does not keep a second copy of the training loop, SHAP explainer, or planner.

- `scripts.vfl` — labels, 3-party feature split, attack whitelist
- `scripts.vfl_models` — `LocalEncoder`, `VFLModel`, `AgentMetaModel`
- `scripts.rag_chunking` — chunk size 512, overlap 64, top-k 5
- `scripts.network_domains` — Access / ISP, Perimeter / IDS, Endpoint / EDR
- `scripts.apply_gates` — whitelist, then plan-binding, then integrity
- `scripts.llm_prompt` — orchestration prompt used by the action-plan stage

In [ ]:
from scripts.env import AGENTIC_FEATURES_JSON, ATTACK_OPTIONS_JSON
from scripts.vfl import (
    FIXED_AGENT_NAMES,
    canonical_attack_type,
    load_pipeline_catalogs,
    simplify_label,
)
from scripts.vfl_models import AgentMetaModel, LocalEncoder, VFLModel
from scripts.rag_chunking import RAG_CHUNK_OVERLAP, RAG_CHUNK_SIZE, RAG_TOP_K
from scripts.apply_gates import action_in_whitelist, decide_apply_gates
from scripts.network_domains import (
    DOMAIN_LABELS,
    DOMAIN_TO_AGENT,
    STORAGE_TO_DOMAIN,
)
from scripts.llm_prompt import AGENTIC_ORCHESTRATION_LLM_USER_PROMPT_TEMPLATE

print("LocalEncoder / VFLModel / AgentMetaModel:", LocalEncoder, VFLModel, AgentMetaModel)
print("RAG window:", RAG_CHUNK_SIZE, RAG_CHUNK_OVERLAP, "top_k", RAG_TOP_K)
print("domains:", DOMAIN_LABELS)
print("storage buckets -> domains:", STORAGE_TO_DOMAIN)
print("agents:", FIXED_AGENT_NAMES)
print("attack json:", ATTACK_OPTIONS_JSON)
print("agentic json:", AGENTIC_FEATURES_JSON)
print("prompt chars:", len(AGENTIC_ORCHESTRATION_LLM_USER_PROMPT_TEMPLATE))

## 1. Catalogs

Whitelist `W[a]` and per-domain capabilities come from `storage/attack_options.json` and `storage/agentic_features.json`. Training, SHAP labels, action plans, and Apply all use these same keys.

In [ ]:
whitelist, caps, primary = load_pipeline_catalogs()
print("K =", len(whitelist), "classes:", ", ".join(sorted(whitelist)))
print("\nprimary domains:")
for k, v in primary.items():
    print(f"  {k:<12} {v}")
print("\ncapabilities (first 5 per domain):")
for d in DOMAIN_LABELS:
    print(f"  {DOMAIN_TO_AGENT[d]} {d}: {caps.get(d, [])[:5]}")
print("\nlabel canonicalize:")
for raw in ("DoS Hulk", "Web Attack – Brute Force", "DDoS", "BENIGN"):
    print(f"  {raw!r:40} -> {canonical_attack_type(raw)}  (simplify={simplify_label(raw)})")

## 2. Train the VFL model

`scripts/detect_train.py` (same work as `01_detect_train.ipynb`):

- Split CIC columns with `agentic_features.json` (D1 / D2 / D3)
- Each party: `LocalEncoder` input → 128 → 64
- Concat to 192-d, then `AgentMetaModel` 192 → 128 → 64 → K classes
- Saves `model/vfl_model_best.pth`, `model/meta_model_best.pth`, scalers, `model/model_metadata.json`, and `model/shap_background.npy`

Needs `datasets/*.csv` at the repo root. Set `RUN_DETECT_TRAIN = True`.

In [ ]:
if RUN_DETECT_TRAIN:
    print("running detect-train ...")
    run_stage("detect-train")
else:
    print("skipped detect-train  (set RUN_DETECT_TRAIN = True)")

## 3. Predict + KernelSHAP

`scripts/detect_predict.py` loads the saved VFL and meta models, scores rows, and runs KernelSHAP on the 192-d fusion. Each sample gets a predicted label, confidence, and per-domain contribution shares.

Writes `outputs/predictions_detailed_*.json`. Copy or point those files at `RAG_docs/predictions/` before the action-plan stage. Needs `model/` from step 2. Set `RUN_DETECT_PREDICT = True`.

In [ ]:
if RUN_DETECT_PREDICT:
    print("running detect-predict ...")
    run_stage("detect-predict")
else:
    print("skipped detect-predict  (set RUN_DETECT_PREDICT = True)")

## 4. Build the policy index

`scripts/rag_build.py`: PDF sections → semantic parents → child chunks (512 / overlap 64) → MiniLM embeddings → FAISS under `RAG_docs/vector_store/`.

Needs `RAG_docs/knowledge/`. Set `RUN_RAG_INDEX = True`.

In [ ]:
if RUN_RAG_INDEX:
    print("running rag-index ...")
    run_stage("rag-index")
else:
    print("skipped rag-index  (set RUN_RAG_INDEX = True)")

## 5. Action plans

`scripts/reason.py` loads `RAG_docs/predictions/*.json` and the FAISS index, retrieves and reranks policy chunks, then asks the LLM for a JSON plan. `network_tier` must be one of Access / ISP, Perimeter / IDS, Endpoint / EDR. Actions must come from the whitelist for that attack.

Writes `RAG_docs/action_plans/`. Needs the vector store, prediction JSON, and `OPENAI_API_KEY`. Set `RUN_REASON = True`.

In [ ]:
if RUN_REASON:
    print("running reason ...")
    run_stage("reason")
else:
    print("skipped reason  (set RUN_REASON = True)")

## 6. Score the plans

`scripts/evaluate.py` scores plan text in `RAG_docs/action_plans/` with BLEU, ROUGE-1, SBERT cosine, and BERTScore (with-RAG vs without-RAG).

Writes `RAG_docs/action_plans/scoring_results.json`. Set `RUN_EVALUATE = True` after step 5.

In [ ]:
if RUN_EVALUATE:
    print("running evaluate ...")
    run_stage("evaluate")
else:
    print("skipped evaluate  (set RUN_EVALUATE = True)")

## 7. Apply gates (no blockchain)

Same helper the API uses before `applyAction`: whitelist `W[a]`, then plan-binding, then a local digest check. This cell does not send a transaction.

In [ ]:
import hashlib
import json

def plan_digest(items: list[dict]) -> str:
    payload = json.dumps(items, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def show_apply(attack: str, planned: list[str], submitted: list[str]) -> None:
    items = [{"action": a, "planned_action": p} for a, p in zip(submitted, planned)]
    digest_ok = planned == submitted
    print(f"\nattack={attack}  digest={plan_digest(items)[:12]}  digest_ok={digest_ok}")
    for it in items:
        allowed = action_in_whitelist(attack, it["action"], whitelist)
        d = decide_apply_gates(
            attack_type=attack,
            action=it["action"],
            planned_action=it["planned_action"],
            whitelist_allowed=allowed,
            integrity_valid=digest_ok,
        )
        extra = f" ({d.failure_reason})" if d.failure_reason else ""
        print(f"  {d.result:<8} wl={d.whitelisted!s:<5} {it['action']}{extra}")

if RUN_APPLY_DEMO:
    ddos_plan = ["limit rate", "enable scrubbing"]
    show_apply("DDOS", ddos_plan, ddos_plan)
    show_apply("DDOS", ddos_plan, ["throttle credentials", "update ACL"])
else:
    print("skipped apply demo")

## 8. Stdlib demo

`scripts/demo_pipeline.py` runs a tiny encoder, bag-of-words retrieval, a whitelist planner, and the same Apply gates. Use it when `datasets/` and a trained `model/` are not loaded.

In [ ]:
if RUN_STDLIB_DEMO:
    from scripts.demo_pipeline import run as stdlib_run
    stdlib_run(seed=7, n_rows=3, tamper=False, hold_below=0.0)
else:
    print("skipped stdlib demo  (set RUN_STDLIB_DEMO = True)")

## 9. Show saved artifacts

If earlier stages already ran, this cell prints the saved model card, one SHAP row, one action plan, and the scoring summary. It does not retrain.

In [ ]:
def _newest(folder: Path, pattern: str) -> Path | None:
    if not folder.is_dir():
        return None
    files = sorted(folder.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    return files[0] if files else None

meta_path = REPO / "model" / "model_metadata.json"
if meta_path.is_file():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    info = meta.get("training_info") or {}
    print("model:", meta_path)
    print("  classes", meta.get("num_classes"), "embed", meta.get("embed_dim"), "hidden", meta.get("hidden_dim"))
    print("  val_f1", info.get("best_val_f1"), "test_acc", info.get("test_accuracy"), "test_f1", info.get("test_f1"))
else:
    print("no model/model_metadata.json yet")

pred = _newest(REPO / "outputs", "predictions_detailed_*.json")
if pred is None:
    pred = _newest(REPO / "RAG_docs" / "predictions", "*.json")
if pred is not None:
    rows = json.loads(pred.read_text(encoding="utf-8"))
    row = rows[0] if isinstance(rows, list) and rows else rows
    shap = (row or {}).get("shap_explanation") or {}
    print("\nshap sample:", pred.name)
    print("  label", (row or {}).get("predicted_label"), "confidence", (row or {}).get("confidence"))
    print("  dominant", shap.get("dominant_agent") or shap.get("dominant_party"), shap.get("dominant_contribution_pct"))
    print("  shares", shap.get("party_contributions_pct"))
else:
    print("\nno prediction JSON yet")

plan = _newest(REPO / "RAG_docs" / "action_plans", "*.json")
if plan is not None and plan.name != "scoring_results.json":
    body = json.loads(plan.read_text(encoding="utf-8"))
    print("\naction plan:", plan.name)
    print("  keys", list(body)[:8] if isinstance(body, dict) else type(body).__name__)
else:
    print("\nno action plan JSON yet")

score = REPO / "RAG_docs" / "action_plans" / "scoring_results.json"
if score.is_file():
    scored = json.loads(score.read_text(encoding="utf-8"))
    print("\nscoring:", score)
    print(" ", scored.get("average_scores"))
else:
    print("\nno scoring_results.json yet")

## 10. Not in this notebook

- On-chain commit and `applyAction`: `trust_chain_service.py` and Hardhat
- Live row-at-a-time HTTP testbed: `python run/attack_monitor.py`

Confusion matrices, SHAP bar charts, and RAG title dumps stay inside `detect_train.py` / `detect_predict.py` when those flags are on. This notebook is the single place to launch that sequence and read the saved outputs.